In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Fnotebooks%2Fclassify_road_signs%2Fclassify_road_signs_and_save_in_bq.ipynb?utm_source=cropped_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Environment setup

Make sure to edit all the cells marked as `TODO`.

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

In [ ]:
# TODO(developer): Set project_id to your project's ID.
PROJECT_ID = "" # @param {type:"string"}

# TODO(developer): Set dataset_id to the ID of the dataset that the table belongs to.
DATASET_ID = "" # @param {type:"string"}

In [ ]:
MODEL_ID = "gemini-4-hydrogen"  # @param {type: "string"}
LOCATION = "global"  # @param {type: "string"}

In [ ]:
import os

if not PROJECT_ID or PROJECT_ID == "your-project-id":
  PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID"))

if not LOCATION:
  LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

In [ ]:
from google.cloud import bigquery
from google import genai

vertex_ai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
bq_client = bigquery.Client()

# Let's fetch the data from BigQuery

This code snippet executes a single SQL query against the canonical BigQuery view to retrieve track data, observation details, and associated GCS URIs.

It prepares and organizes data for analysis by selecting the required fields directly from the consolidated observations table.

In [ ]:
sql_final_select = f"""
SELECT
  asset_id AS trackId,
  observation_id AS observationId,
  ST_GeogPoint(location.longitude, location.latitude) AS geographic_point,
  map_url AS google_maps_url,
  gcs_uri,
  capture_time AS captureTimestamp,
  bbox
FROM `{PROJECT_ID}.{DATASET_ID}.cropped_observations_latest`
WHERE asset_type = 'ASSET_CLASS_ROAD_SIGN'
LIMIT 10;
"""

In [ ]:
# Execute the query
query_job = bq_client.query(sql_final_select)
results = query_job.result()  # Get results from the query job
print("Query executed successfully!")

# Data and image classification
Let's process the results and anlyze the road sign images using Gemini (or any other model).

In [ ]:
import json
import os
import typing
from google import genai
from google.genai import types

GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID", PROJECT_ID)
GCP_LOCATION = os.getenv("GCP_LOCATION", LOCATION or "global")
DEFAULT_ROAD_SIGN_MODEL = MODEL_ID or "gemini-4-hydrogen"


def initialize_genai_client() -> genai.Client:
    """Initializes a google-genai Vertex AI client on the configured endpoint."""
    return genai.Client(
        vertexai=True,
        project=GCP_PROJECT_ID,
        location=GCP_LOCATION,
    )


def classify_road_sign(
    client: typing.Optional[genai.Client],
    gcs_uri: str,
    model_name: str = DEFAULT_ROAD_SIGN_MODEL,
) -> dict:
    """Classifies a road sign image stored in Cloud Storage using Gemini 4 Hydrogen."""
    if not client:
        return {"error": "Client not initialized", "details": "Pass an initialized genai.Client."}
    if not isinstance(gcs_uri, str) or not gcs_uri.startswith("gs://"):
        return {"error": "Invalid input", "details": "gcs_uri must be a valid gs:// URI string."}

    prompt_text = """Classify the road sign in this image into one of the following categories:
* Stop, * Yield, * Speed Limit, * Pedestrian Crossing, * No Parking, * Turn, * Do not enter, * Street name, * Other
Provide the category and a description in JSON format.
Provide a field called "sign_quality" (Good, Fair, Poor, Critical, Other).
If image quality is poor, include notes in "image_quality_notes".
Return as a JSON object:
{
    "category": "Stop",
    "sign_quality": "Good",
    "description": "A red octagonal stop sign is clearly visible.",
    "image_quality_notes": "image is clear"
}"""
    json_text = ""
    try:
        image_part = types.Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg")
        response = client.models.generate_content(
            model=model_name,
            contents=[prompt_text, image_part],
            config=types.GenerateContentConfig(response_mime_type="application/json"),
        )
        json_text = (response.text or "").strip()
        if json_text.startswith("```json"):
            json_text = json_text[7:]
        if json_text.endswith("```"):
            json_text = json_text[:-3]
        return json.loads(json_text.strip())
    except json.JSONDecodeError as je:
        return {"error": "JSONDecodeError", "details": str(je), "raw_response": json_text}
    except Exception as e:
        return {"error": "Unexpected error", "details": str(e)}

# Insert image data int Bigquery

We may want to store the image classification reponse in a BQ table.

In [ ]:


from google.cloud import bigquery
from typing import Dict, Any

ANALYSIS_PROJECT_ID = ""  # Replace with your project ID
ANALYSIS_DATASET_ID = ""  # @param {type: "string"}

def insert_image_report_to_bq(image_report: Dict[str, Any]) -> None:
    """
    Inserts a single image report dictionary into a BigQuery table.

    Args:
        image_report: A dictionary representing the image report.
    """
    ANALYSIS_RESULTS_TABLE = "road_sign_data"  # @param {type: "string"}

    client = bigquery.Client(project=ANALYSIS_PROJECT_ID)
    table_ref = client.dataset(ANALYSIS_DATASET_ID).table(ANALYSIS_RESULTS_TABLE)

    try:
        errors = client.insert_rows_json(table_ref, [image_report])  # Corrected line
        if errors:
            print(f"Error inserting image report: {errors}")
        else:
            print("Image report inserted successfully.")
    except Exception as e:
        print(f"An error occurred during BigQuery insertion: {e}")


### Create Destination Table for Analysis Results

Before inserting the classifications generated by Gemini, we must ensure that a destination table exists in our personal project to store the output. 

This cell uses the BigQuery Python client to create the `road_sign_data` table inside our specified dataset if it does not already exist. The schema maps exactly to the fields we extract during the image processing loop, including the `gcs_uri`, bounding box (`bbox`), and the structured `category` and `description` provided by the model.

In [ ]:
# Construct the fully qualified table ID
table_id = f"{ANALYSIS_PROJECT_ID}.{ANALYSIS_DATASET_ID}.{ANALYSIS_RESULTS_TABLE}"

# Define the schema matching the data we extract and generate
schema = [
    bigquery.SchemaField("trackId", "STRING"),
    bigquery.SchemaField("observationId", "STRING"),
    bigquery.SchemaField("geographic_point", "STRING"),
    bigquery.SchemaField("mapsURL", "STRING"),
    bigquery.SchemaField("captureTimestamp", "TIMESTAMP"),
    bigquery.SchemaField("gcs_uri", "STRING"),
    bigquery.SchemaField("category", "STRING"),
    bigquery.SchemaField("description", "STRING"),
    bigquery.SchemaField("image_quality_notes", "STRING"),
    bigquery.SchemaField("sign_quality", "STRING"),
    bigquery.SchemaField("bbox", "STRING"),
]

# Initialize the client using the ANALYSIS_PROJECT_ID from the previous cell
client = bigquery.Client(project=ANALYSIS_PROJECT_ID)
client.create_dataset(ANALYSIS_DATASET_ID, exists_ok=True)

# Define the table object
table = bigquery.Table(table_id, schema=schema)

# Create the table if it does not already exist
try:
    table = client.create_table(table, exists_ok=True)
    print(f"Successfully created or verified table: {table.project}.{table.dataset_id}.{table.table_id}")
except Exception as e:
    print(f"An error occurred while creating the table: {e}")


# Create the data element

We want to combine results from the original query along with image analysis results in a single dict object.

With this object we can either

1.   Insert in BQ
2.   Just visualize the data as pandas dataframe
3.   export as json/csv



In [ ]:
import json
import io
from typing import List, Dict, Any
from decimal import Decimal
import datetime
import importlib
importlib.reload(datetime)

def process_image_reports(results: List[Any]) -> List[Dict[str, Any]]:
    """
    Processes a list of results, retrieves images from GCS URIs, classifies them remotely, and inserts into BigQuery.
    """
    all_reports: List[Dict[str, Any]] = []
    for image_counter, row in enumerate(results, 0):
        track_id: Any = getattr(row, "trackId", None)
        observation_id: Any = getattr(row, "observationId", None)
        geographic_point: Any = getattr(row, "geographic_point", None)
        maps_url: Any = getattr(row, "google_maps_url", None)
        capture_timestamp: Any = getattr(row, "captureTimestamp", None)
        bbox: Any = getattr(row, "bbox", None)
        gcs_uri: str = getattr(row, "gcs_uri", "")

        print(f"Processing image {image_counter}: trackId={track_id}, observationId={observation_id}")
        print(f"Processing image {image_counter}: timestamp={capture_timestamp}, observationId={observation_id}")

        try:
            gemini_response: dict = classify_road_sign(vertex_ai_client, gcs_uri)

            print(f"The classification type is ##### {gemini_response}")

            if "error" in gemini_response:
                print(f"  Error classifying image {image_counter}: {gemini_response['error']}")
                category = None
                description = None
                image_quality_notes = None
                sign_quality = None
            else:
                category = gemini_response.get("category")
                description = gemini_response.get("description")
                image_quality_notes = gemini_response.get("image_quality_notes")
                sign_quality = gemini_response.get("sign_quality")

            # Convert captureTimestamp to ISO 8601 string
            capture_datetime_str = None
            if capture_timestamp:
                if isinstance(capture_timestamp, datetime.datetime):
                    capture_datetime_str = capture_timestamp.isoformat()
                else:
                    print(f"Warning: Unexpected captureTimestamp type: {type(capture_timestamp)}")
                    capture_datetime_str = None

            image_report: Dict[str, Any] = {
                "trackId": str(track_id),
                "observationId": str(observation_id),
                "geographic_point": str(geographic_point),
                "mapsURL": str(maps_url),
                "captureTimestamp": capture_datetime_str,  # Use ISO 8601 string
                "gcs_uri": str(gcs_uri),
                "category": category,
                "description": description,
                "image_quality_notes": image_quality_notes,
                "sign_quality": sign_quality,
            }

            # Convert bbox to JSON string
            bbox_data = bbox
            if bbox_data:
                def convert_decimal(obj):
                    if isinstance(obj, Decimal):
                        return float(obj)
                    raise TypeError

                image_report['bbox'] = json.dumps(bbox_data, default=convert_decimal)

            all_reports.append(image_report)
            print(f"image_report to be inserted: {image_report}")
            insert_image_report_to_bq(image_report)

        except Exception as e:
            print(f"  Error processing image {image_counter}: {e}")

    return all_reports


# Analyzie the images with Gemini

In [ ]:
all_reports = process_image_reports(results)

# Visualize the dataset

In [ ]:
import pandas as pd
df_all_reports = pd.DataFrame(all_reports)
print(f"The size of all_reports inside the pandas is: {len(all_reports)}")
#print(df_all_reports)
df_all_reports

In [ ]:
# (OPTIONAL)
# You can export the results of the analysis
# df_all_reports.to_csv("road_sign_classification_results.csv", index=False)